# 02 — Results (imports everything from `src/`)
**Owner: Member 5 — Evaluation & Deployment Engineer**.

Presents the experiment log, baseline comparison and final test metrics — exactly what the Week-14 report and the live demo rely on.

In [ ]:
import sys
from pathlib import Path
if not Path("configs").exists(): sys.path.insert(0, str(Path.cwd().parent))
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from configs import config as cfg

log = pd.read_csv(cfg.EXPERIMENTS_DIR / 'experiment_log.csv')
log.tail(10)

## 1. Baselines vs neural models (val split)

In [ ]:
base = Path(cfg.EXPERIMENTS_DIR / 'baseline_results.csv')
if base.exists():
    print(pd.read_csv(base).to_string(index=False))

full = log[(log.config_tag != 'smoke') & log.max_per_class.isna() & ~log.config_tag.isin(['final_seed', 'seed_parent'])]
best = full.sort_values('val_macro_f1').groupby('model').tail(1)
print(best[['model', 'run_id', 'lr', 'batch_size', 'seed',
            'val_accuracy', 'val_macro_f1', 'params']].to_string(index=False))

## 2. Learning curves of the reference runs

In [ ]:
for run in best['run_id']:
    hist = Path(cfg.RUNS_DIR / f'{run}_history.csv')
    if hist.exists():
        df = pd.read_csv(hist)
        fig, ax = plt.subplots(1, 2, figsize=(9, 3))
        ax[0].plot(df.accuracy, label='train'); ax[0].plot(df.val_accuracy, label='val')
        ax[1].plot(df.loss, label='train'); ax[1].plot(df.val_loss, label='val')
        for a in ax: a.legend(); a.grid(alpha=0.3); a.set_xlabel('epoch')
        fig.suptitle(run); plt.show()

## 3. Final test metrics (used ONCE, at the end)

In [ ]:
mfile = cfg.EXPERIMENTS_DIR / 'final_evaluation' / 'comparison.json'
if mfile.exists():
    print(json.dumps(json.loads(mfile.read_text()), indent=2))
else:
    print('run `python -m src.evaluate --latest` first')

### Takeaways for the report

Fill these after Milestone 4 — every number must come from the files above, not from memory:

1. Improvement over baselines: `<final val macro-F1>` vs `<MLP baseline>`.
2. Best model on validation (and why): `<model / config>`.
3. Most common confusion pair and the proposed fix.
4. Honest limitations: `<overfitting signs, weak classes, compute budget>`.